# 7 - Structure Functions

The second-order structure function,

$$S_2(\ell) = \langle\, [f(x+\ell) - f(x)]^2 \,\rangle,$$

measures how a field $f$ (e.g. a velocity residual map) decorrelates with
separation $\ell$. It rises from zero at small lag to a plateau of $2\sigma^2$
once points are uncorrelated, and the lag of the rollover encodes the
correlation scale.

`eddy` computes a *2D* structure function on a polar `(radius, azimuth)` grid,
can resolve it by reference radius, and provides tools to denoise, collapse, and
reduce it. This tutorial covers the basics on a self-contained synthetic field.
To apply it to real data, see
`eddy.momentmap.momentmap.compute_structure_function_stack`, which deprojects a
sky map onto the polar grid first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

from eddy.structurefunction import (
    StructureFunction2D,
    StructureFunction2DStack,
    structure_function_ensemble,
)

## A Synthetic Field

We build a correlated field on a polar grid (**axis 0 = radius, axis 1 =
azimuth**) by smoothing white noise. Using a larger azimuthal than radial
smoothing makes it anisotropic, which we will see in the structure function.

In [ ]:
rng = np.random.default_rng(42)
n_r, n_phi = 120, 240
dr = 0.02     # arcsec per radial pixel
dphi = 1.5    # degrees per azimuthal pixel

field = gaussian_filter(rng.standard_normal((n_r, n_phi)), sigma=(4, 12), mode="nearest")
field /= field.std()

plt.imshow(field, origin="lower", aspect="auto")
plt.xlabel("azimuth [pix]"); plt.ylabel("radius [pix]"); plt.title("synthetic field");

## The Global Structure Function

`StructureFunction2D.from_array` with `ref_i=-1` pools every pair in the field
into a single global $S_2$. The result carries the 2D surface (`S2`), the radial
and azimuthal slices (`S2_x`, `S2_y`), and the azimuthally-averaged profile
(`S2_i`).

In [ ]:
sf = StructureFunction2D.from_array(field, dx=dr, dy=dphi, ref_i=-1, azimuthal_axis="y")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
im = axes[0].imshow(sf.S2, origin="lower", extent=sf.extent, aspect="auto")
fig.colorbar(im, ax=axes[0])
axes[0].set_title("2D $S_2$")
axes[0].set_xlabel("azimuthal lag [deg]"); axes[0].set_ylabel("radial lag [arcsec]")

axes[1].plot(sf.lags_x, sf.S2_x, label="radial ($S_{2,x}$)")
axes[1].plot(sf.lags_i, sf.S2_i, label="azimuthal avg ($S_{2,i}$)")
axes[1].axhline(2.0, ls=":", color="k", label=r"$2\sigma^2$")
axes[1].set_xlabel("lag [arcsec]"); axes[1].set_ylabel("$S_2$"); axes[1].legend();

## Radius-Resolved: The Stack

For a radius-dependent analysis, compute $S_2$ at a sequence of reference radii.
`StructureFunction2DStack.from_array` does this on a bare polar array (pass the
radial coordinate as `x_axis` so the reference radii map to the right rows).
The radial / azimuthal / anisotropy *heatmaps* show how the structure varies
with radius.

In [ ]:
ref_rs = np.linspace(0.3, 2.0, 12)
stack = StructureFunction2DStack.from_array(
    field, ref_rs, x_axis=np.arange(n_r) * dr, dx=dr, dy=dphi, azimuthal_axis="y",
)

X, Y, C = stack.calculate_azimuthal_heatmap()
plt.pcolormesh(X, Y, C)
plt.xlabel("azimuthal lag [deg]"); plt.ylabel(r"reference radius [arcsec]")
plt.colorbar(label="$S_2$"); plt.title("azimuthal heatmap");

## Removing a Noise Model

Structure functions of independent components add, so a noise contribution can
be subtracted at the $S_2$ level. Build a **mean noise stack** by combining many
noise realizations (`combine` is pair-count-weighted), then `subtract` it from
the observed stack.

`structure_function_ensemble` turns a 3D array of realizations into a *list* of
results (one per field), without averaging them.

In [ ]:
# many noise realizations -> ensemble of stacks -> mean noise stack
noise = gaussian_filter(rng.standard_normal((20, n_r, n_phi)), sigma=(0, 2, 2), mode="nearest")
noise_stacks = structure_function_ensemble(
    noise, mode="stack", ref_rs=ref_rs, x_axis=np.arange(n_r) * dr, dx=dr, dy=dphi,
)
mean_noise = noise_stacks[0].combine(noise_stacks[1:])

clean = stack.subtract(mean_noise)   # denoised stack

## Collapsing a Stack to a Global $S_2$

`collapse()` pools the radius axis (pair-count-weighted) into a single
`StructureFunction2D` — the radial/azimuthal slices match a direct global
`ref_i=-1` result exactly.

In [ ]:
global_s2 = clean.collapse()
print("plateau (~ 2 sigma^2):", round(global_s2.plateau(), 3))

## Ensembles for Uncertainties

To estimate how much $S_2$ scatters across realizations of a field, build an
ensemble (`mode="global"` here for one global $S_2$ per field) and reduce across
the realization axis with percentiles.

In [ ]:
fields = gaussian_filter(rng.standard_normal((50, n_r, n_phi)), sigma=(0, 4, 12), mode="nearest")
fields /= fields.std(axis=(1, 2), keepdims=True)

ens = structure_function_ensemble(fields, mode="global", dx=dr, dy=dphi, azimuthal_axis="y")
S2x = np.array([s.S2_x for s in ens])           # (N, n_lag)

lags = ens[0].lags_x
lo, hi = np.percentile(S2x, [16, 84], axis=0)
plt.plot(lags, S2x.mean(0), label="mean")
plt.fill_between(lags, lo, hi, alpha=0.3, label="16-84%")
plt.xlabel("radial lag [arcsec]"); plt.ylabel("$S_2$"); plt.legend();

## Scalar Summaries

Convenience methods reduce a result (or stack) to scalars: `plateau`
(the $2\sigma^2$ asymptote), `half_power_lag` (a model-free correlation scale),
and `reliability_weight` (a per-annulus weight for cross-radius averages). The
stack exposes per-annulus versions (`plateaus`, `half_power_lags`,
`reliability_weights`).

In [ ]:
print("global plateau              :", round(global_s2.plateau(), 3))
print("global radial half-power lag :", round(global_s2.half_power_lag("x"), 3), "arcsec")
print("per-annulus radial lags      :", np.round(clean.half_power_lags("x"), 3))

## Where to Go Next

- `StructureFunction2D` / `StructureFunction2DStack`: full API (subtract,
  combine, collapse, heatmaps, spiral fitting via `fit_spiral`).
- `eddy.momentmap.momentmap.compute_structure_function_stack`: build a stack
  directly from a sky map, with deprojection.
- `gaussian_beam_s2`: the analytic beam-noise $S_2$ for the subtraction above
  when you only have beam properties.